# 03 — Findings: The Story in 7 Charts
**Group P33** — Mika Ohayon, Akshaya Somasundaram, Harini Champooranan

**Research question:** Do U.S. jobs that are highly exposed to AI actually show the job and wage losses that AI-displacement headlines predict, and is that different for high-skill vs low-skill jobs?

**The short answer:** AI exposure is highest in **well-paid, high-skill jobs**, and those jobs are doing fine. The only early sign of displacement is in **lower-skill jobs that are highly exposed to AI** (mostly office and clerical work). They lost workers from 2022 to 2025 while similar jobs grew.

| Part | Question | Charts |
|---|---|---|
| A | Who is exposed to AI? (2025) | 1, 2, 3 |
| B | What actually happened from 2022 to 2025? | 4, 5, 6 |
| C | Do the models confirm it? | 7 |

**Reading the charts:** every chart uses the same two colors. **Blue = higher-skill jobs** (managers, professionals, tech, healthcare). **Pink = lower-skill jobs** (service, sales, office, trades, production, transport). *AI exposure* is a standardized score: 0 is an average job, +1 is well above average, and −1 is well below.

Requires `finalized_data.csv` from notebook 01.

In [1]:
# If a package is missing, uncomment and run once:
# %pip install pandas numpy altair scikit-learn statsmodels

import warnings
import numpy as np
import pandas as pd
import altair as alt
import statsmodels.formula.api as smf
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

warnings.filterwarnings('ignore')
alt.data_transformers.disable_max_rows()
pd.set_option('display.width', 140)

HI, LO = 'Higher-skill (SOC 11–29)', 'Lower-skill (SOC 31–53)'
TIER = alt.Scale(domain=[HI, LO], range=['#1f5fbf', '#d6337a'])
tier_color = lambda legend=True: alt.Color('skill_tier:N', scale=TIER,
                                           legend=alt.Legend(title='Skill tier', orient='top') if legend else None)

df = pd.read_csv('finalized_data.csv', dtype={'soc_major': str, 'OCC_CODE': str})

# change 2022 -> 2025 (drop very small jobs whose counts are mostly sampling noise)
chg = df.dropna(subset=['A_MEDIAN_22', 'TOT_EMP_22']).copy()
chg = chg[(chg['TOT_EMP_22'] >= 1000) & (chg['TOT_EMP'] >= 1000)]
chg['emp_growth'] = np.log(chg['TOT_EMP'] / chg['TOT_EMP_22'])
chg['wage_growth'] = np.log(chg['A_MEDIAN'] / chg['A_MEDIAN_22'])
chg['emp_pct'] = 100 * (chg['TOT_EMP'] / chg['TOT_EMP_22'] - 1)
chg['wage_pct'] = 100 * (chg['A_MEDIAN'] / chg['A_MEDIAN_22'] - 1)
chg['log_wage_22'] = np.log(chg['A_MEDIAN_22'])
chg['log_emp_22'] = np.log10(chg['TOT_EMP_22'])
chg['exposure_level'] = pd.qcut(chg['AIOE'], 3, labels=['Low', 'Medium', 'High']).astype(str)

print(f'2025 analysis: {len(df)} occupations | 2022 → 2025 change: {len(chg)} occupations')
print(f"Typical job 2022 → 2025: workers {chg['emp_pct'].median():+.1f}%, pay {chg['wage_pct'].median():+.1f}%")

2025 analysis: 670 occupations | 2022 → 2025 change: 661 occupations
Typical job 2022 → 2025: workers +2.9%, pay +13.1%


---
# Part A — Who is exposed to AI? (2025)

---
### Visual 1 — Which kinds of jobs are most exposed to AI?
**What you're looking at:** Each bar is a job category (for example Legal or Construction). Bar length is the category's average AI exposure. Bars to the right of the dashed line are more exposed than the average job, and bars to the left are less exposed.

**What to notice:** Every blue (higher-skill) category is above the average line. Most pink (lower-skill) categories sit at the bottom. The two exceptions are **Office & Admin** and **Sales**, which are lower-skill but highly exposed. We come back to them in Part B.

> **Key takeaway:** AI exposure isn't spread evenly. It's concentrated in **office-based, information-heavy work** (Computer & Math, Legal, Business & Financial) and lowest in **physical, hands-on work** (Construction, Cleaning).

*How to interact:* hover over a bar for details. Click a skill tier in the legend to highlight it, and double-click to reset.

In [2]:
groups = (df.groupby(['major_group', 'skill_tier'], as_index=False)
            .agg(avg_exposure=('AIOE', 'mean'), jobs=('AIOE', 'size'), workers=('TOT_EMP', 'sum')))

pick = alt.selection_point(fields=['skill_tier'], bind='legend')
bars = alt.Chart(groups).mark_bar().encode(
    x=alt.X('avg_exposure:Q', title='Average AI exposure  (← less exposed  |  more exposed →)'),
    y=alt.Y('major_group:N', sort='-x', title=None),
    color=tier_color(),
    opacity=alt.condition(pick, alt.value(1), alt.value(0.2)),
    tooltip=[alt.Tooltip('major_group:N', title='Job category'),
             alt.Tooltip('avg_exposure:Q', title='Avg exposure', format='+.2f'),
             alt.Tooltip('jobs:Q', title='Occupations'),
             alt.Tooltip('workers:Q', title='Workers', format=',.0f')],
).add_params(pick)
zero = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(strokeDash=[4, 4], color='black').encode(x='x:Q')
zero_lbl = alt.Chart(pd.DataFrame({'x': [0], 't': ['Average job']})).mark_text(
    align='left', dx=4, dy=-6, fontStyle='italic', y=0).encode(x='x:Q', text='t:N')

visual1 = (bars + zero + zero_lbl).properties(
    width=560, height=430, title='Visual 1 — Average AI exposure by job category')
visual1

alt.LayerChart(...)

---
### Visual 2 — Do more exposed jobs pay more or less?
**What you're looking at:** Each dot is one occupation. Further right means more exposed to AI, and higher up means higher median pay (2025). Each skill tier has its own trend line.

**What to notice:** Both lines slope **upward**, so more exposed jobs pay more. The blue line is much steeper.

> **Key takeaway:** This is the opposite of the common belief that AI threatens low-paid work. In higher-skill jobs, each step up in exposure goes with **about 21% higher pay**, but in lower-skill jobs only **about 4%**. Because the two tiers behave so differently, we analyze them separately for the rest of the notebook.

*How to interact:* hover over a dot to see the job. Scroll to zoom, drag to pan, and click the legend to show one tier.

In [3]:
fits, labels = [], []
for tier, g in df.groupby('skill_tier'):
    slope, icpt = np.polyfit(g['AIOE'], g['log_wage'], 1)
    for x in (g['AIOE'].min(), g['AIOE'].max()):
        fits.append({'skill_tier': tier, 'AIOE': x, 'A_MEDIAN': np.exp(icpt + slope * x)})
    labels.append({'skill_tier': tier, 'AIOE': g['AIOE'].max(), 'A_MEDIAN': np.exp(icpt + slope * g['AIOE'].max()),
                   'label': f'{100 * (np.exp(slope) - 1):+.0f}% pay per +1 exposure'})

pick = alt.selection_point(fields=['skill_tier'], bind='legend')
y = alt.Y('A_MEDIAN:Q', scale=alt.Scale(type='log', domain=[28000, 380000]), title='Median annual pay, 2025 (log scale)', axis=alt.Axis(format='$,.0f'))
dots = alt.Chart(df).mark_circle(size=45, clip=True).encode(
    x=alt.X('AIOE:Q', title='AI exposure  (← less  |  more →)'), y=y, color=tier_color(),
    opacity=alt.condition(pick, alt.value(0.55), alt.value(0.05)),
    tooltip=[alt.Tooltip('occupation:N', title='Job'), alt.Tooltip('AIOE:Q', title='Exposure', format='+.2f'),
             alt.Tooltip('A_MEDIAN:Q', title='Median pay', format='$,.0f')],
).add_params(pick, alt.selection_interval(bind='scales'))
lines = alt.Chart(pd.DataFrame(fits)).mark_line(size=4).encode(x='AIOE:Q', y=y, color=tier_color(False))
lab = pd.DataFrame(labels)
lab['label'] = np.where(lab['skill_tier'] == HI, 'Higher-skill: ', 'Lower-skill: ') + lab['label']
lab['ypix'] = np.where(lab['skill_tier'] == HI, 14, 34)
text = alt.Chart(lab).mark_text(align='left', x=10, fontWeight='bold', fontSize=13).encode(
    y=alt.Y('ypix:Q', scale=None), text='label:N', color=tier_color(False))

visual2 = (dots + lines + text).properties(width=640, height=420,
                                           title='Visual 2 — More AI-exposed jobs tend to pay more')
visual2

alt.LayerChart(...)

---
### Visual 3 — Which specific jobs are most and least exposed?
**What you're looking at:** The 10 most exposed and the 10 least exposed occupations. Each bar's label shows the job's median pay.

**What to notice:** The top 10 are almost all analytical, desk-based professions, mostly paying **$85k–$155k**, such as actuaries, judges, budget analysts and accountants. The bottom 10 are physical jobs paying much less, such as fitness trainers and construction helpers.

> **Key takeaway:** "AI exposure" measures how much of a job is **reading, analyzing, writing and deciding**. Those happen to be well-paid tasks.

*How to interact:* use the buttons above the chart to switch between the most and least exposed jobs. Hover over a bar for details.

In [4]:
top = df.nlargest(10, 'AIOE').assign(view='Most exposed')
bottom = df.nsmallest(10, 'AIOE').assign(view='Least exposed')
ranked = pd.concat([top, bottom])
ranked['pay_label'] = ranked['A_MEDIAN'].map(lambda v: f'${v / 1000:,.0f}k')

view = alt.param(value='Most exposed', bind=alt.binding_radio(options=['Most exposed', 'Least exposed'], name='Show: '))
base = alt.Chart(ranked).transform_filter(alt.datum.view == view)
bars = base.mark_bar().encode(
    x=alt.X('AIOE:Q', title='AI exposure'),
    y=alt.Y('occupation:N', sort=alt.EncodingSortField('AIOE', order='descending'), title=None,
            axis=alt.Axis(labelLimit=320)),
    color=tier_color(),
    tooltip=[alt.Tooltip('occupation:N', title='Job'), alt.Tooltip('AIOE:Q', title='Exposure', format='+.2f'),
             alt.Tooltip('A_MEDIAN:Q', title='Median pay', format='$,.0f'),
             alt.Tooltip('TOT_EMP:Q', title='Workers', format=',.0f')])
pay = base.mark_text(align='left', dx=4).encode(
    x=alt.X('AIOE:Q'), y=alt.Y('occupation:N', sort=alt.EncodingSortField('AIOE', order='descending')),
    text='pay_label:N')

visual3 = (bars + pay).add_params(view).properties(
    width=520, height=320, title='Visual 3 — The 10 most / least AI-exposed jobs (label = median pay)')
visual3

alt.LayerChart(...)

---
# Part B — What actually happened from 2022 to 2025?
Part A shows **who is exposed**, but exposure isn't the same as job loss. To test displacement, we compare each job in **May 2022** (just before ChatGPT launched) with **May 2025**.

To keep the comparison fair, we split all jobs into three equal-sized groups by AI exposure (**Low, Medium, High**) and compare each group with the typical job. Groups with fewer than 10 jobs are left out because they're too small to trust.

In [5]:
def by_level(value_col):
    t = (chg.groupby(['skill_tier', 'exposure_level'], as_index=False)
            .agg(median_change=(value_col, 'median'), jobs=('AIOE', 'size'),
                 workers_2022=('TOT_EMP_22', 'sum'), workers_2025=('TOT_EMP', 'sum')))
    return t[t['jobs'] >= 10]


def level_chart(table, typical, y_title, title):
    lvl = ['Low', 'Medium', 'High']
    bars = alt.Chart(table).mark_bar().encode(
        x=alt.X('exposure_level:N', sort=lvl, title='AI exposure level', axis=alt.Axis(labelAngle=0)),
        xOffset=alt.XOffset('skill_tier:N', sort=[HI, LO]),
        y=alt.Y('median_change:Q', title=y_title),
        color=tier_color(),
        tooltip=[alt.Tooltip('skill_tier:N', title='Tier'), alt.Tooltip('exposure_level:N', title='Exposure'),
                 alt.Tooltip('median_change:Q', title='Median change %', format='+.1f'),
                 alt.Tooltip('jobs:Q', title='Occupations'),
                 alt.Tooltip('workers_2025:Q', title='Workers 2025', format=',.0f')])
    labels = bars.mark_text(dy=-8, fontWeight='bold').encode(
        text=alt.Text('median_change:Q', format='+.1f'), color=alt.value('black'))
    ref = alt.Chart(pd.DataFrame({'y': [typical]})).mark_rule(strokeDash=[5, 4], color='black').encode(y='y:Q')
    ref_lbl = alt.Chart(pd.DataFrame({'y': [typical], 't': [f'Typical job: {typical:+.1f}%']})).mark_text(
        align='right', x=520, dy=-7, fontStyle='italic').encode(y='y:Q', text='t:N')
    zero = alt.Chart(pd.DataFrame({'y': [0]})).mark_rule(color='grey').encode(y='y:Q')
    return (bars + labels + zero + ref + ref_lbl).properties(width=520, height=340, title=title)

---
### Visual 4 — Did more exposed jobs lose workers?
**What you're looking at:** The **median change in number of workers** from 2022 to 2025, by AI exposure level and skill tier. The dashed line is the typical job (+2.9%).

**What to notice:** Focus on the **pink bars**. Lower-skill jobs with **High** exposure **shrank by about 9%**, while lower-skill jobs with Medium exposure grew. Higher-skill exposed jobs (blue) **grew about 6–7%**, above the typical job.

> **Key takeaway:** This is the **only place we see displacement**: lower-skill jobs that are highly exposed to AI are losing workers. Highly exposed high-skill jobs are growing.

*How to interact:* hover over a bar for the number of jobs and workers in that group.

In [6]:
emp_tbl = by_level('emp_pct')
visual4 = level_chart(emp_tbl, chg['emp_pct'].median(), 'Median change in workers, 2022 → 2025 (%)',
                      'Visual 4 — Change in number of workers by AI exposure')
visual4

alt.LayerChart(...)

---
### Visual 5 — Did more exposed jobs get smaller raises?
**What you're looking at:** Same layout as Visual 4, but for the **median change in pay** from 2022 to 2025. The dashed line is the typical job (+13.1%, mostly inflation).

**What to notice:** All the bars are about the same height, between **+11% and +15%**. Exposure makes only small differences in pay growth.

> **Key takeaway:** AI exposure **didn't affect pay**. Exposed jobs got the same raises as everyone else. So far, the AI effect shows up in **how many people are employed**, not in what they're paid.

*How to interact:* hover over a bar for details.

In [7]:
wage_tbl = by_level('wage_pct')
visual5 = level_chart(wage_tbl, chg['wage_pct'].median(), 'Median change in pay, 2022 → 2025 (%)',
                      'Visual 5 — Change in pay by AI exposure')
visual5

alt.LayerChart(...)

---
### Visual 6 — Which exposed lower-skill jobs shrank the most?
**What you're looking at:** The 10 **lower-skill, highly AI-exposed** jobs (each with at least 20,000 workers in 2022) that lost the largest share of their workers.

**What to notice:** These are mostly **clerical and phone-based** jobs: telemarketers (−39%), order clerks, loan interviewers, switchboard operators, bill collectors, account clerks, medical transcriptionists. Their tasks are routine information processing that software and AI tools can take over. Together these 10 jobs went from about **1.0 million to 750,000 workers (−25%)**.

> **Key takeaway:** This puts names to Visual 4. The shrinking jobs are exactly the kind of routine office work you'd expect AI and automation to replace first.

*How to interact:* hover over a bar to see the worker counts in 2022 and 2025.

In [8]:
shrank = (chg[(chg['skill_tier'] == LO) & (chg['exposure_level'] == 'High') & (chg['TOT_EMP_22'] >= 20000)]
          .nsmallest(10, 'emp_pct').copy())
shrank['label'] = shrank['emp_pct'].map(lambda v: f'{v:.0f}%')

bars = alt.Chart(shrank).mark_bar(color='#d6337a').encode(
    x=alt.X('emp_pct:Q', title='Change in workers, 2022 → 2025 (%)'),
    y=alt.Y('occupation:N', sort='x', title=None, axis=alt.Axis(labelLimit=340)),
    tooltip=[alt.Tooltip('occupation:N', title='Job'), alt.Tooltip('major_group:N', title='Category'),
             alt.Tooltip('TOT_EMP_22:Q', title='Workers 2022', format=',.0f'),
             alt.Tooltip('TOT_EMP:Q', title='Workers 2025', format=',.0f'),
             alt.Tooltip('emp_pct:Q', title='Change %', format='+.1f')])
labels = bars.mark_text(align='left', dx=4, color='white', fontWeight='bold').encode(text='label:N')

visual6 = (bars + labels).properties(width=480, height=320,
                                     title='Visual 6 — Lower-skill, highly exposed jobs that shrank the most')
print(f"These 10 jobs went from {shrank['TOT_EMP_22'].sum():,.0f} to {shrank['TOT_EMP'].sum():,.0f} workers "
      f"({shrank['TOT_EMP'].sum() / shrank['TOT_EMP_22'].sum() - 1:+.0%}).")
visual6

These 10 jobs went from 1,005,730 to 749,370 workers (-25%).


alt.LayerChart(...)

---
# Part C — Do the models confirm it?
Charts can mislead, for example when a group happens to contain a few unusual jobs. So we train models on **one question**:

> **Can a job's AI exposure predict how much its number of workers changed from 2022 to 2025?**

**Target:** change in workers, 2022 → 2025 (log change).
**Inputs:** AI exposure, skill tier, **exposure × skill tier** (lets exposure work differently in each tier), and each job's 2022 pay and size (controls, so we don't confuse AI with "small jobs are noisier" or "low-paid jobs behave differently").

**Three models**, each scored with **5-fold cross-validation**: the data is split into 5 parts, and each part is used once as unseen test data.
| Model | What it is | Why we use it |
|---|---|---|
| Linear regression (OLS) | A straight-line model | Easy to interpret; gives a p-value for each input |
| Random Forest | 300 decision trees averaged together | Can find curves and thresholds a line misses |
| Gradient Boosting | 300 small trees, each fixing the last one's errors | Usually the most accurate model on tables like ours |

In [9]:
FEATURES = ['AIOE', 'high_skill', 'AIOE_x_skill', 'log_wage_22', 'log_emp_22']
X, y = chg[FEATURES], chg['emp_growth']
folds = KFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'Linear regression (OLS)': LinearRegression(),
    'Random Forest': RandomForestRegressor(n_estimators=300, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=3, random_state=42),
}
rows = []
for name, m in models.items():
    train_r2 = m.fit(X, y).score(X, y)
    cv_r2 = cross_val_score(m, X, y, cv=folds, scoring='r2').mean()
    rows.append({'Model': name, 'R² on data it trained on': round(train_r2, 2),
                 'R² on unseen data (5-fold CV)': round(cv_r2, 2)})
results = pd.DataFrame(rows)
results

,Model,R² on data it trained on,R² on unseen data (5-fold CV)
0,Linear regression (OLS),0.11,0.05
1,Random Forest,0.63,-0.00
2,Gradient Boosting,0.63,-0.07


**How to read this table:** R² is the share of the differences in job growth the model explains (1 = all, 0 = none). The column that matters is **unseen data**.

> **Key takeaway:**
> - **All three models score near 0 on unseen data.** Exposure and skill can't predict *which* individual jobs grew or shrank. Job growth from 2022 to 2025 was driven mostly by other things (post-COVID recovery, interest rates, layoffs).
> - **The tree models overfit.** They score 0.6–0.7 on data they trained on but about 0 on new data, which means they memorized noise. The simpler linear model is the honest choice.
> - A low R² doesn't mean there is *no* effect. It means AI is a **small** factor so far. The next step tests whether that small effect is real.

In [10]:
ols = smf.ols('emp_growth ~ AIOE * high_skill + log_wage_22 + log_emp_22', data=chg).fit(cov_type='HC1')

# effect of +1 exposure on worker growth, separately for each tier (percentage points)
b, V = ols.params, ols.cov_params()
lo_eff, lo_se = b['AIOE'], np.sqrt(V.loc['AIOE', 'AIOE'])
hi_eff = b['AIOE'] + b['AIOE:high_skill']
hi_se = np.sqrt(V.loc['AIOE', 'AIOE'] + V.loc['AIOE:high_skill', 'AIOE:high_skill'] + 2 * V.loc['AIOE', 'AIOE:high_skill'])
from scipy.stats import norm
effects = pd.DataFrame({
    'skill_tier': [HI, LO],
    'effect': [100 * hi_eff, 100 * lo_eff],
    'low': [100 * (hi_eff - 1.96 * hi_se), 100 * (lo_eff - 1.96 * lo_se)],
    'high': [100 * (hi_eff + 1.96 * hi_se), 100 * (lo_eff + 1.96 * lo_se)],
    'p_value': [2 * (1 - norm.cdf(abs(hi_eff / hi_se))), 2 * (1 - norm.cdf(abs(lo_eff / lo_se)))],
})
effects['result'] = np.where(effects['p_value'] < 0.05, 'Statistically significant', 'Not significant')
effects.round(3)

,skill_tier,effect,low,high,p_value,result
0,Higher-skill (SOC 11–29),0.650,-3.108,4.408,0.735,Not significant
1,Lower-skill (SOC 31–53),-2.779,-5.142,-0.417,0.021,Statistically significant


---
### Visual 7 — How much does AI exposure change job growth, once we control for other factors?
**What you're looking at:** The result of the linear regression. Each dot is the estimated effect of **one step more AI exposure** on a job's worker growth, in percentage points. The line through each dot is the 95% confidence range. If that range crosses the dashed zero line, we can't rule out "no effect."

**What to notice:** The **pink (lower-skill)** range sits entirely **below zero**. The **blue (higher-skill)** range crosses zero.

> **Key takeaway:** Even after controlling for pay and job size, **more AI exposure means slower job growth in lower-skill jobs: about −2.8 percentage points per step of exposure (p = 0.02)**. For higher-skill jobs there's **no measurable effect**. This confirms Visual 4 statistically. The difference between the two tiers isn't significant yet (p = 0.11), so treat it as an **early signal**.

*How to interact:* hover over a dot for the exact estimate, its range and the p-value.

In [11]:
dot = alt.Chart(effects).mark_point(filled=True, size=220).encode(
    x=alt.X('effect:Q', title='Effect of +1 AI exposure on worker growth (percentage points)'),
    y=alt.Y('skill_tier:N', title=None, sort=[HI, LO]),
    color=tier_color(),
    tooltip=[alt.Tooltip('skill_tier:N', title='Tier'), alt.Tooltip('effect:Q', title='Effect (pts)', format='+.2f'),
             alt.Tooltip('low:Q', title='95% low', format='+.2f'), alt.Tooltip('high:Q', title='95% high', format='+.2f'),
             alt.Tooltip('p_value:Q', title='p-value', format='.3f'), alt.Tooltip('result:N', title='Result')])
ci = alt.Chart(effects).mark_rule(size=4).encode(
    x='low:Q', x2='high:Q', y=alt.Y('skill_tier:N', sort=[HI, LO]), color=tier_color(False))
lbl = alt.Chart(effects).mark_text(dy=-18, fontWeight='bold').encode(
    x='effect:Q', y=alt.Y('skill_tier:N', sort=[HI, LO]),
    text=alt.Text('effect:Q', format='+.1f'), color=tier_color(False))
zero = alt.Chart(pd.DataFrame({'x': [0]})).mark_rule(strokeDash=[5, 4], color='black').encode(x='x:Q')
zero_lbl = alt.Chart(pd.DataFrame({'x': [0], 't': ['No effect']})).mark_text(
    align='left', dx=4, y=8, fontStyle='italic').encode(x='x:Q', text='t:N')

visual7 = (ci + dot + lbl + zero + zero_lbl).properties(
    width=560, height=200, title='Visual 7 — Effect of AI exposure on job growth (with 95% confidence range)')
visual7

alt.LayerChart(...)

---
# Conclusions
**1. Who is exposed?** AI exposure is highest in **well-paid, high-skill, desk-based jobs** (Visuals 1–3), not the low-wage work people usually worry about.

**2. Is displacement happening?** So far, only in one place. From 2022 to 2025, **lower-skill jobs that are highly exposed to AI lost about 9% of their workers** while the typical job grew 3% (Visual 4). These are mostly routine clerical and phone-based jobs (Visual 6). The regression confirms this effect is statistically significant (Visual 7).

**3. High-skill vs low-skill?** The tiers behave differently. Highly exposed high-skill jobs **grew** and earn more. Exposure looks more like AI *helping* those workers than replacing them.

**4. Pay?** AI exposure had **no effect on raises** in either tier (Visual 5).

**5. How big is the AI effect overall?** Small so far. The models can't predict which individual jobs grew (R² ≈ 0), so AI is one factor among many.

**Bottom line:** exposure is highest in the jobs many of us are training for, but those jobs are growing. The early losses are in routine lower-skill office work.

### Limitations
- **Three years is a short window**, and other things changed at the same time (post-COVID labor market, 2023 tech layoffs). This is a pattern consistent with displacement, not proof AI caused it.
- **Skill tier is based on job category**, not each job's actual education requirement.
- **157 occupations have no AI exposure score**, because the exposure data uses older job codes.
- The **tier difference isn't statistically significant yet** (p = 0.11). It's worth re-testing as new BLS data comes out.